<p> <center><img src="https://datascientest.fr/train/assets/logo_datascientest.png" style="height:200px"> </p>

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> Analyse et prédiction de la variabilité de la production solaire à partir de données ouvertes de la région PACA </h1></center>
<center><h2> Collecte des données athmosphériques </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">


Nous avons précédemment collecté les données de **production d'énergie solaire** et les données sur la **position du soleil**. Or, parmi les connaissances de l'homme du métier, d'autres données que celles concernant la position du soleil pourraient permettre d'expliquer notre variable cible et être utiles à notre modélisation.

Ainsi, dans ce notebook, nous nous intéressons aux *rayonnements solaires effectivements reçus* au niveau des panneaux solaires et donc à des données concernant l'**athmosphère**.

Ces données sont par exemple disponibles dans un jeu de données `CAMS solar radiation time-serie` mis gracieusement à disposition par Copernicus (programme
d’observation de la Terre de l’Union européenne - https://www.copernicus.eu/fr) via son service de surveillance de l’atmosphère (CAMS - https://atmosphere.copernicus.eu/).

Pour chacun des points d'intérêts identifiés pour la production d'énergie solaire, nous récupèrerons les variables :
 - **GHI** : irradiance horizontale globale *(mesure globale de la lumière solaire reçue sur un plan horizontal au niveau du sol)* ;
 - **BHI** : irradiance horizontale directe *(mesure des rayonnements solaires directs reçus sur un plan horizontal au niveau du sol)* ;
 - **DHI** : irradiance horizontale diffuse *(mesure des rayonnements solaires indirects, diffusés dans l'athmosphère reçus sur un plan horizontal au niveau du sol)* ;
 - **BNI** : irradiance normale directe *(mesure des rayonnements solaires directs reçus sur un plan perpendiculaire aux rayons solaires et se déplaçant avec ce dernier)*.

Pour chacune de ces variables, nous récupèreront également leur valeur théorique maximale (pour un ciel dégagé).


# I - Récupération des données des étapes précédentes

Nous commencons par récupérer les données de production collectées aux étapes précédentes depuis le répertoire du projet.


In [1]:
# Chemin du dataset de donnees astronomiques
input_datasets = '../../data/local_data/02_Collecte_datasets/02_Astronomie/output/astro_2020_2024.csv'

# Chemin du dataset des communes sélectionnées 
communes = '../../data/local_data/01_Clustering/output/best_communes_geo_energy.csv'

# Chemin vers le répertoire où seront stockés les résultats
output_datasets = '../../data/local_data/02_Collecte_datasets/03_Athmosphere/output/'

On importe les librairies nécessaires pour manipuler nos données :

In [2]:
# On importe les librairies dont on a besoin pour traiter les datasets
import pandas as pd
import numpy as np

On charge le jeu de données de l'étape précédente :

In [3]:
# On récupère le dataset contenant les données de production et les données astronomiques
df_astro = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])
display(df_astro.head())
df_astro.dtypes


/tmp/ipykernel_22624/3721904188.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_astro = pd.read_csv(input_datasets, parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),CRU_azimuth,CRU_altitude,SEL_azimuth,SEL_altitude,SVT_azimuth,SVT_altitude,BRA_azimuth,BRA_altitude,EYG_azimuth,EYG_altitude
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,335.596867,-67.454606,336.734433,-67.160776,337.561520,-68.097845,335.241040,-68.046190,333.265411,-67.469007
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,353.817769,-68.883102,354.738150,-68.485180,356.336005,-69.329403,353.945722,-69.500739,351.451534,-69.119526
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,12.883689,-68.564280,13.430816,-68.099948,15.637215,-68.757908,13.536657,-69.141123,10.834111,-69.009045
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,30.261125,-66.570892,30.451783,-66.089459,32.859299,-66.517353,31.240402,-67.054165,28.719513,-67.163582
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,44.631373,-63.284229,44.594428,-62.821307,46.883423,-63.030485,45.699703,-63.656435,43.545408,-63.957709


datetime_utc           datetime64[ns, UTC]
Périmètre                           object
Nature                              object
Consommation                       float64
Solaire                            float64
Ech. physiques                     float64
Stockage batterie                   object
Déstockage batterie                 object
TCO Solaire (%)                    float64
TCH Solaire (%)                    float64
CRU_azimuth                        float64
CRU_altitude                       float64
SEL_azimuth                        float64
SEL_altitude                       float64
SVT_azimuth                        float64
SVT_altitude                       float64
BRA_azimuth                        float64
BRA_altitude                       float64
EYG_azimuth                        float64
EYG_altitude                       float64
dtype: object

# II - Lieux retenus

On récupère les coordonnées des cinq points significatifs que l'on a précédemment déterminé.


In [4]:
df_communes = pd.read_csv(communes)
display(df_communes.head())
print(df_communes.dtypes)

,cluster_geo,best_commune,code_insee,lat,lon,energie_totale,poids,prefix
0,2,Cruis,4065,44.0845,5.8397,20356525.0,0.22,CRU
1,4,Saint-Étienne-le-Laus,5140,44.5075,6.1616,325158.0,0.06,SEL
2,0,Saint-Vallier-de-Thiey,6130,43.6994,6.8516,344281.0,0.07,SVT
3,1,Bras,83021,43.4723,5.9558,10603661.0,0.29,BRA
4,3,Eygalières,13034,43.7638,4.9554,1510927.0,0.36,EYG


cluster_geo         int64
best_commune       object
code_insee          int64
lat               float64
lon               float64
energie_totale    float64
poids             float64
prefix             object
dtype: object


# III - Collecte des données athmosphériques

Le jeu de données **CAMS solar radiation time-serie** (https://ads.atmosphere.copernicus.eu/datasets/cams-solar-radiation-timeseries?tab=overview) est accessible via une API par l'intermédiaire de la librairie cdsapi.


On importe cdsapi avec d'autres librairies utiles pour la collecte :

In [5]:
# On importe les librairies dont on a besoin pour accéder aux données
import yaml
import os
import cdsapi
import time

L'accès à l'API se fait via un compte : on charge donc les données correspondantes pour accéder à l'API :

In [6]:
# URL de l'API
url = "https://ads.atmosphere.copernicus.eu/api"

# API Token
with open('../../data/local_data/02_Collecte_datasets/03_Athmosphere/input/cams_access', 'r') as f:
    key = f.read().splitlines()[0] # On lit la première ligne du fichier qui contient une clé


Pour faciliter la collecte des données pour chacun de nos points d'intérêts, on crée une fonction qui interroge l'API en fonction des coordonnées géographiques d'un lieu (latitude et longitude) et des dates de début et de fin de la période à observer.

Un jeu de données est alors enregistré au niveau d'un chemin transmis à la fonction.


In [7]:
def retrieve_ghi (latitude, longitude, date_debut, date_fin, base_chemin) :
    # On crée le client de l'API
    client = cdsapi.Client(url=url, key=key)
    
    # Nom du jeu de données qui nous intéresse
    dataset = "cams-solar-radiation-timeseries"
    
    # Formulation de la requête
    requete = {"sky_type": "observed_cloud",
               "location": {"longitude": longitude, "latitude": latitude},
               "altitude": ["0"],
               "date": [date_debut + "/" + date_fin],
               "time_step": "15minute", # '30minute' n'existe pas on passe directement à un intervalle d'une heure
               "time_reference": "universal_time",
               "data_format": "csv"}
    
    target = base_chemin + "_" +date_debut + "_" + date_fin + ".csv"
    
    # Envoi de la requête
    fname = client.retrieve(dataset, requete, target)
    

On détermine, à partir du jeu de données précédemment collecté, la date de début et la date de fin de la période couverte.

In [8]:
# On détermine la plage de dates à requêter
# En heures UTC le dataset de départ commence le 31 décembre 2019 et se termine le 31 décembre 2024

date_debut = df_astro.loc[0,'datetime_utc'].strftime('%Y-%m-%d')
date_fin = df_astro.loc[df_astro.shape[0]-1,'datetime_utc'].strftime('%Y-%m-%d')

print("Date de début :", date_debut)
print("Date de fin :", date_fin)

Date de début : 2019-12-31
Date de fin : 2024-12-31


On interroge l'API pour collecter les données athmosphérique pour chaque point d'intérêt :

In [9]:
# On interroge l'API pour récupérer les données CAMS de chaque ville
# Pour chaque ville
for ville in df_communes['prefix'] :
    # On affiche où on en est rendu dans les villes
    print(ville + "...")
    
    # Le chemin de base du dataset temporaire sera :
    base_chemin = output_datasets + "CAMS_" + ville # La fonction de récupération ajoutera les dates et l'extension

    latitude = df_communes.loc[df_communes['prefix']==ville, 'lat'].iloc[0]
    longitude = df_communes.loc[df_communes['prefix']==ville, 'lon'].iloc[0]
    
    # Envoi de la requete
    retrieve_ghi(latitude, longitude,
                 date_debut, date_fin,
                 base_chemin)
    
    # On indique que la requete est terminée
    print("Ok pour", ville, '\n')
    
    # On patiente avant la requête suivante
    time.sleep(10)


CRU...


2026-02-06 13:51:55,592 INFO Request ID is bfdbe00f-794a-467b-bb6c-7d0eed386dd2
2026-02-06 13:51:55,690 INFO status has been updated to accepted
2026-02-06 13:52:09,285 INFO status has been updated to running
2026-02-06 13:52:45,648 INFO status has been updated to successful


Ok pour CRU
SEL...


2026-02-06 13:53:00,256 INFO Request ID is 6bf3d195-b34b-4481-b871-848073ca28fb
2026-02-06 13:53:00,333 INFO status has been updated to accepted
2026-02-06 13:53:14,240 INFO status has been updated to running
2026-02-06 13:53:50,521 INFO status has been updated to successful


Ok pour SEL
SVT...


...

Ok pour SVT
BRA...


2026-02-06 13:55:08,343 INFO Request ID is 1743413d-9809-4e3d-8329-244c3a1ccb53
2026-02-06 13:55:08,427 INFO status has been updated to accepted
2026-02-06 13:55:22,001 INFO status has been updated to running
2026-02-06 13:55:58,304 INFO status has been updated to successful


Ok pour BRA
EYG...


2026-02-06 13:56:12,463 INFO Request ID is 11cef06d-f52e-4c12-a533-59e88ad7fe19
2026-02-06 13:56:12,522 INFO status has been updated to accepted
2026-02-06 13:56:21,034 INFO status has been updated to running
2026-02-06 13:57:02,784 INFO status has been updated to successful


Ok pour EYG


# IV - Aggrégation des nouvelles données collectées

Les données collectées se présentent pour le moment sous la forme de fichiers csv enregistrés dans le Drive du projet, à raison d'un fichier par point d'intérêt.

Pour aggréger les nouvelles variables explicatives, on va :

 - Charger les fichiers sous forme de DataFrame Pandas ;
 - Renommer les variables en les faisant commencer par le sigle du point d'intérêt ;
 - Fusionner les jeux de données sur la base de la variable temporelle.


On charge sous forme de DataFrame Pandas les jeux de données qu'on vient de collecter :

In [10]:
# On crée une fonction lambda pour obtenir la date au bon format
# (on conserve la borne de début de l'intervalle de temps)
f = lambda x: pd.to_datetime(x.split('/')[0]).tz_localize("UTC")

In [13]:
# On charge les dataset sous forme de DataFrame pour pouvoir les traiter
# Attention : environ 10 minutes d'exécution...
print("Cruis...")
cams_CRU = pd.read_csv(output_datasets + 'CAMS_CRU_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Saint-Étienne-le-Laus...')
cams_SEL = pd.read_csv(output_datasets + 'CAMS_SEL_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Saint-Vallier-de-Thiey...')
cams_SVT = pd.read_csv(output_datasets + 'CAMS_SVT_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})

print('Bras...')
cams_BRA = pd.read_csv(output_datasets + 'CAMS_BRA_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})
print("Eygalières...")
cams_EYG = pd.read_csv(output_datasets + 'CAMS_EYG_2019-12-31_2024-12-31.csv',
                      sep=';', header=42,
                      converters={'# Observation period': f})
print("=> OK")

Cruis...
Saint-Étienne-le-Laus
Saint-Vallier-de-Thiey
Bras
Eygalières...
OK


On réalise une première observation des jeux de données collectés :

In [14]:
# On commence par regarder à quoi ressemble un dataset cams
display(cams_CRU.head())
display(cams_CRU.describe())
print(cams_CRU.info())


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
0,2019-12-31 00:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 00:15:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2019-12-31 00:30:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2019-12-31 00:45:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2019-12-31 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
count,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000,175488.000000
mean,77.755296,56.941334,45.531930,11.409404,85.715740,45.993977,30.537864,15.456113,57.820092,0.978015
std,98.561004,75.791909,63.332389,14.703329,96.251817,67.071796,53.286388,23.586732,83.829046,0.094220
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
50%,2.681750,0.703850,0.063350,0.612600,1.248400,0.494900,0.000000,0.451300,0.000000,1.000000
75%,150.655975,108.472650,85.880775,20.895650,192.007125,81.157225,42.042975,23.361600,123.476800,1.000000
max,308.452000,253.079600,232.761400,112.624400,258.484300,253.079600,232.761400,134.651200,258.484300,1.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175488 entries, 0 to 175487
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype              
---  ------                --------------   -----              
 0   # Observation period  175488 non-null  datetime64[ns, UTC]
 1   TOA                   175488 non-null  float64            
 2   Clear sky GHI         175488 non-null  float64            
 3   Clear sky BHI         175488 non-null  float64            
 4   Clear sky DHI         175488 non-null  float64            
 5   Clear sky BNI         175488 non-null  float64            
 6   GHI                   175488 non-null  float64            
 7   BHI                   175488 non-null  float64            
 8   DHI                   175488 non-null  float64            
 9   BNI                   175488 non-null  float64            
 10  Reliability           175488 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(10)
memory usage:

Pour parcourir plus facilement les datasets on utilise un dictionnaires ayant pour clé le sigle correspondant au point d'intérêt du dataset :

In [15]:
# 'Cruis' 'Saint-Étienne-le-Laus' 'Saint-Vallier-de-Thiey'  'Bras'  'Eygalières'
all_cams = {'CRU' : cams_CRU,
            'SEL' : cams_SEL,
            'SVT' : cams_SVT,
            'BRA' : cams_BRA,
            'EYG' : cams_EYG}

# On affiche quelques valeurs après 10h30 du matin
for df in all_cams.values():
    display(df.iloc[42:51, :])
    

,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,127.9681,93.9969,79.7676,14.2293,219.3695,93.9969,79.7676,14.2293,219.3695,1.0
43,2019-12-31 10:45:00+00:00,131.5526,97.1360,82.7308,14.4052,221.3223,97.1360,82.7308,14.4052,221.3223,1.0
44,2019-12-31 11:00:00+00:00,134.1630,99.4364,84.9078,14.5286,222.7285,99.4364,84.9078,14.5286,222.7285,1.0
45,2019-12-31 11:15:00+00:00,135.7882,100.8832,86.2814,14.6018,223.6242,100.8832,86.2814,14.6018,223.6242,1.0
46,2019-12-31 11:30:00+00:00,136.4211,101.4670,86.8410,14.6260,224.0310,101.4670,86.8410,14.6260,224.0310,1.0
47,2019-12-31 11:45:00+00:00,136.0591,101.1840,86.5821,14.6019,223.9572,101.1840,86.5821,14.6019,223.9572,1.0
48,2019-12-31 12:00:00+00:00,134.7037,100.0313,85.5051,14.5263,223.3960,100.0313,85.5051,14.5263,223.3960,1.0
49,2019-12-31 12:15:00+00:00,132.3607,98.0161,83.6177,14.3984,222.3308,98.0161,83.6177,14.3984,222.3308,1.0
50,2019-12-31 12:30:00+00:00,129.0402,95.1549,80.9356,14.2193,220.7347,95.1549,80.9356,14.2193,220.7347,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,125.9731,93.6346,80.5471,13.0875,225.0217,93.6346,80.5471,13.0875,225.0217,1.0
43,2019-12-31 10:45:00+00:00,129.4494,96.6949,83.4551,13.2398,226.8875,96.6949,83.4551,13.2398,226.8875,1.0
44,2019-12-31 11:00:00+00:00,131.9575,98.9171,85.5725,13.3446,228.2243,98.9171,85.5725,13.3446,228.2243,1.0
45,2019-12-31 11:15:00+00:00,133.4866,100.2871,86.8832,13.4038,229.0667,100.2871,86.8832,13.4038,229.0667,1.0
46,2019-12-31 11:30:00+00:00,134.0303,100.7959,87.3771,13.4187,229.4350,100.7959,87.3771,13.4187,229.4350,1.0
47,2019-12-31 11:45:00+00:00,133.5861,100.4401,87.0503,13.3898,229.3367,100.4401,87.0503,13.3898,229.3367,1.0
48,2019-12-31 12:00:00+00:00,132.1560,99.2158,85.8942,13.3216,228.7387,99.2158,85.8942,13.3216,228.7387,1.0
49,2019-12-31 12:15:00+00:00,129.7461,97.1301,83.9170,13.2131,227.6228,97.1301,83.9170,13.2131,227.6228,1.0
50,2019-12-31 12:30:00+00:00,126.3667,94.2034,81.1454,13.0580,225.9886,94.2034,81.1454,13.0580,225.9886,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,131.1590,94.5056,78.4712,16.0344,210.5552,94.5056,78.4712,16.0344,210.5552,1.0
43,2019-12-31 10:45:00+00:00,134.5034,97.3787,81.1246,16.2542,212.2648,97.3787,81.1246,16.2542,212.2648,1.0
44,2019-12-31 11:00:00+00:00,136.8639,99.4104,82.9942,16.4162,213.4131,99.4104,82.9942,16.4162,213.4131,1.0
45,2019-12-31 11:15:00+00:00,138.2305,100.5875,84.0652,16.5223,214.0309,100.5875,84.0652,16.5223,214.0309,1.0
46,2019-12-31 11:30:00+00:00,138.5973,100.9022,84.3290,16.5732,214.1346,100.9022,84.3290,16.5732,214.1346,1.0
47,2019-12-31 11:45:00+00:00,137.9626,100.3524,83.7834,16.5689,213.7277,100.3524,83.7834,16.5689,213.7277,1.0
48,2019-12-31 12:00:00+00:00,136.3294,98.9590,82.4766,16.4824,212.9138,98.9590,82.4766,16.4824,212.9138,1.0
49,2019-12-31 12:15:00+00:00,133.7044,96.7305,80.4179,16.3126,211.6732,96.7305,80.4179,16.3126,211.6732,1.0
50,2019-12-31 12:30:00+00:00,130.0990,93.6631,77.5794,16.0837,209.8583,93.6631,77.5794,16.0837,209.8583,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,131.4660,94.2978,78.3354,15.9624,209.6990,92.1881,74.9243,17.2639,200.5964,1.0
43,2019-12-31 10:45:00+00:00,135.0570,97.3852,81.2028,16.1824,211.5981,93.5265,74.6596,18.8669,194.5532,1.0
44,2019-12-31 11:00:00+00:00,137.6635,99.6347,83.2890,16.3457,212.9268,98.4517,81.2562,17.1955,207.7092,1.0
45,2019-12-31 11:15:00+00:00,139.2744,101.0313,84.5770,16.4543,213.7200,101.0313,84.5770,16.4543,213.7200,1.0
46,2019-12-31 11:30:00+00:00,139.8826,101.5661,85.0566,16.5094,213.9978,101.5661,85.0566,16.5094,213.9978,1.0
47,2019-12-31 11:45:00+00:00,139.4857,101.2351,84.7238,16.5114,213.7667,101.2351,84.7238,16.5114,213.7667,1.0
48,2019-12-31 12:00:00+00:00,138.0853,100.0559,83.6305,16.4255,213.1476,100.0559,83.6305,16.4255,213.1476,1.0
49,2019-12-31 12:15:00+00:00,135.6874,98.0357,81.7845,16.2512,212.1250,98.0357,81.7845,16.2512,212.1250,1.0
50,2019-12-31 12:30:00+00:00,132.3022,95.1715,79.1505,16.0210,210.5436,95.1715,79.1505,16.0210,210.5436,1.0


,# Observation period,TOA,Clear sky GHI,Clear sky BHI,Clear sky DHI,Clear sky BNI,GHI,BHI,DHI,BNI,Reliability
42,2019-12-31 10:30:00+00:00,128.7457,92.4755,77.1252,15.3503,210.8207,92.4755,77.1252,15.3503,210.8207,1.0
43,2019-12-31 10:45:00+00:00,132.5785,95.7792,80.2260,15.5532,212.9601,95.7792,80.2260,15.5532,212.9601,1.0
44,2019-12-31 11:00:00+00:00,135.4351,98.2533,82.5536,15.6996,214.5187,98.2533,82.5536,15.6996,214.5187,1.0
45,2019-12-31 11:15:00+00:00,137.3035,99.8815,84.0892,15.7923,215.5371,99.8815,84.0892,15.7923,215.5371,1.0
46,2019-12-31 11:30:00+00:00,138.1755,100.6534,84.8206,15.8329,216.0403,100.6534,84.8206,15.8329,216.0403,1.0
47,2019-12-31 11:45:00+00:00,138.0475,100.5639,84.7418,15.8221,216.0397,100.2795,84.2420,16.0375,214.7630,1.0
48,2019-12-31 12:00:00+00:00,136.9200,99.6067,83.8463,15.7604,215.5164,94.1017,74.1831,19.9186,190.6517,1.0
49,2019-12-31 12:15:00+00:00,134.7978,97.7878,82.1410,15.6468,214.4559,90.4277,69.2881,21.1396,180.9015,1.0
50,2019-12-31 12:30:00+00:00,131.6900,95.1254,79.6464,15.4790,212.8480,89.4275,69.7941,19.6333,186.5382,1.0


A ce stade, nous avons des données athmosphériques avec une résolution temporelle de **15 minutes**.

Cependant, le jeu de données dans lequel nous souhaitons ajouter ces données athmosphériques a une résolution temporelle de **30 minutes**.

Or les données que nous avons collectées concernant l'athmosphères sont toutes cumulatives (à l'exception de la fiabilité `Reliability`) : pour avoir les données pour 30 minutes nous devons maintenant sommer les lignes deux à deux.

Pour que la variable `Reliability` reste pertinente, on sa remplace par la moyenne entre le temp *t* et le temp *t-1*.

In [16]:
# On parcours l'ensemble des datasets
for df in all_cams.values():
    # Les colonnes à sommer vont de la deuxième à l'avant dernière (index 1 à -1)

    # On sélectionne les heures +15min et +45min (toutes les lignes impaires)
    df_impair = df.iloc[1:-1:2, 1:]
    
    # On sélectionne les heures +0min et +30min (toutes les lignes paires)
    # La première ligne n'est pas traitée et importe peu car a lieu en pleine nuit quand il n'y a pas de soleil
    df_pair = df.iloc[2::2, 1:]

    # On modifie l'index de df_impair pour pouvoir le sommer à df_pair
    df_impair.set_index(df_pair.index, inplace=True)

    # On met à jour le dataset avec les valeurs sommées
    df.iloc[2::2, 1:] = df_pair.add(df_impair)

    # On divise par deux la variable 'Reliability' (dernière colonne) pour avoir sa moyenne
    df.iloc[2::2, -1] /= 2


On renomme les variables des datasets (à l'exeption de la variable `datetime_utc` qui servira à la fusion) en les faisants débuter par le sigle du point d'intérêt correspondant :

In [17]:
for ville, df in all_cams.items() :
    mon_dico = {'# Observation period' : 'datetime_utc',
                'TOA' : ville + '_TOA',
                'Clear sky GHI' : ville + '_Clear sky GHI',
                'Clear sky BHI' : ville + '_Clear sky BHI',
                'Clear sky DHI' : ville + '_Clear sky DHI',
                'Clear sky BNI' : ville + '_Clear sky BNI',
                'GHI' : ville + '_GHI',
                'BHI' : ville + '_BHI',
                'DHI' : ville + '_DHI',
                'BNI' : ville + '_BNI',
                'Reliability' : ville + '_Reliability'}
    df.rename(mon_dico, axis=1, inplace=True)


On fusionne les datasets en se basant sur la variable `datetime_utc` qui contient la date et l'heure des observations :  

In [18]:
for df in all_cams.values():
    df_astro = df_astro.merge(right=df, on='datetime_utc', how='left')

On affiche le début du jeu de données pour avoir une première visualisation des nouvelles données :

In [19]:
print(df_astro.shape)
display(df_astro.iloc[23:28, :])

(87696, 70)


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
23,2020-01-01 10:30:00+00:00,PACA,Données définitives,5355.0,587.0,3248.0,-,-,10.96,43.74,...,253.2484,181.5259,150.7824,30.7435,419.0294,181.0210,149.8874,31.1336,416.6206,1.0
24,2020-01-01 11:00:00+00:00,PACA,Données définitives,5389.0,637.0,3222.0,-,-,11.82,47.47,...,268.6971,194.6732,162.9588,31.7144,426.8785,173.7657,126.2110,47.5547,330.7245,1.0
25,2020-01-01 11:30:00+00:00,PACA,Données définitives,5510.0,665.0,3325.0,-,-,12.07,49.55,...,276.2950,201.1038,168.8604,32.2434,430.1928,176.9042,126.1173,50.7869,321.3643,1.0
26,2020-01-01 12:00:00+00:00,PACA,Données définitives,5396.0,667.0,3208.0,-,-,12.36,49.70,...,275.9122,200.6609,168.3135,32.3474,429.3926,177.5202,127.8147,49.7055,326.1265,1.0
27,2020-01-01 12:30:00+00:00,PACA,Données définitives,5453.0,650.0,3289.0,-,-,11.92,48.44,...,267.5549,193.3875,161.3886,31.9990,424.5634,173.1829,126.9076,46.2753,333.8453,1.0


# V - Enregistrement des données CAMS collectées

Nous avons terminé la collecte des données explicatives relatives à l'`athmosphère` pour chaque point significatif du point de vue de la production d'énergie solaire, via le jeu de données **CAMS solar radiation time-serie**.

Nous enregistrons notre jeu de données actuel pour clore la troisième partie de notre travail de collecte.

In [20]:
# On enregistre ce dataset contenant les données de production, astronomiques et météo partielle avant ajout d'autres variables
df_astro.to_csv(output_datasets + "cams_2020_2024.csv", index=False)


In [21]:
# Exemple de la manière dont charger ce dataset production+astro final :
df_cams = pd.read_csv(output_datasets + "cams_2020_2024.csv", parse_dates=['datetime_utc'])
display(df_cams.head())
df_cams.dtypes

/tmp/ipykernel_19132/4156042889.py:2: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cams = pd.read_csv(output_datasets + "cams_2020_2024.csv", parse_dates=['datetime_utc'])


,datetime_utc,Périmètre,Nature,Consommation,Solaire,Ech. physiques,Stockage batterie,Déstockage batterie,TCO Solaire (%),TCH Solaire (%),...,EYG_TOA,EYG_Clear sky GHI,EYG_Clear sky BHI,EYG_Clear sky DHI,EYG_Clear sky BNI,EYG_GHI,EYG_BHI,EYG_DHI,EYG_BNI,EYG_Reliability
0,2019-12-31 23:00:00+00:00,PACA,Données définitives,6123.0,0.0,3332.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2019-12-31 23:30:00+00:00,PACA,Données définitives,5907.0,0.0,2837.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2020-01-01 00:00:00+00:00,PACA,Données définitives,5724.0,0.0,2668.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2020-01-01 00:30:00+00:00,PACA,Données définitives,5749.0,0.0,2754.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2020-01-01 01:00:00+00:00,PACA,Données définitives,5700.0,0.0,2886.0,-,-,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


datetime_utc       datetime64[ns, UTC]
Périmètre                       object
Nature                          object
Consommation                   float64
Solaire                        float64
                          ...         
EYG_GHI                        float64
EYG_BHI                        float64
EYG_DHI                        float64
EYG_BNI                        float64
EYG_Reliability                float64
Length: 70, dtype: object